# Reproduce Results

This notebook **loads pre-trained models from disk** and evaluates them on the
train / validation / test splits.  No training happens here.

Metrics reported:
- **ROC-AUC** (`sklearn.metrics.roc_auc_score`)
- **Precision** (`sklearn.metrics.precision_score`)
- **Recall** (`sklearn.metrics.recall_score`)
- **F1** (`sklearn.metrics.f1_score`)

In [1]:
import os
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split

import utils
from utils import (
    load_object,
    get_combined_features,
    evaluate_model,
)

## 1. Data loading and splitting

The exact same random seed and split ratios as `train_models.ipynb` are used
so that train / val / test sets are identical across notebooks.

In [2]:
DATA_PATH = os.path.expanduser("~/Datasets/QuoraQuestionPairs/quora_data.csv")
quora_df = pd.read_csv(DATA_PATH)

A_df, test_df = train_test_split(quora_df, test_size=0.05, random_state=123)
train_df, val_df = train_test_split(A_df,  test_size=0.05, random_state=123)

print(f'train_df.shape = {train_df.shape}')
print(f'val_df.shape   = {val_df.shape}')
print(f'test_df.shape  = {test_df.shape}')

y_train = train_df["is_duplicate"].values
y_val   = val_df["is_duplicate"].values
y_test  = test_df["is_duplicate"].values

train_df.shape = (291897, 6)
val_df.shape   = (15363, 6)
test_df.shape  = (16172, 6)


## 2. Load models and vectorizers

In [3]:
MODELS_DIR = "models"

count_vectorizer    = load_object(os.path.join(MODELS_DIR, "count_vectorizer.pkl"))
tfidf_vectorizer    = load_object(os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl"))
baseline_logistic   = load_object(os.path.join(MODELS_DIR, "baseline_logistic.pkl"))
improved_logistic   = load_object(os.path.join(MODELS_DIR, "sbert_logistic.pkl"))

print("All models and vectorizers loaded successfully.")

All models and vectorizers loaded successfully.


## 3. Feature extraction

- **Baseline**: sparse BoW matrix (CountVectorizer, unigrams)
- **Improved**: BoW + 5 handcrafted similarity features (see `utils.py`)

In [4]:
print("\nExtracting combined (BoW + handcrafted) features...")
X_train_comb = get_combined_features(train_df, count_vectorizer, tfidf_vectorizer)
X_val_comb   = get_combined_features(val_df,   count_vectorizer, tfidf_vectorizer)
X_test_comb  = get_combined_features(test_df,  count_vectorizer, tfidf_vectorizer)
print(f"  train shape: {X_train_comb.shape}")

y_train = train_df["is_duplicate"].values
y_val   = val_df["is_duplicate"].values


Extracting combined (BoW + handcrafted) features...


  train shape: (291897, 149659)


## 4. Evaluation — ROC-AUC, Precision, Recall, F1

In [5]:
rows = []
'''
# Baseline model on all three splits
rows.append(evaluate_model(baseline_logistic, X_train_bow, y_train, "baseline", "train"))
rows.append(evaluate_model(baseline_logistic, X_val_bow,   y_val,   "baseline", "val"))
rows.append(evaluate_model(baseline_logistic, X_test_bow,  y_test,  "baseline", "test"))
'''
# Improved model on all three splits
rows.append(utils.evaluate_model(baseline_logistic, X_train_comb, y_train, "baseline", "train"))
rows.append(utils.evaluate_model(baseline_logistic, X_val_comb,   y_val,   "baseline", "val"))
rows.append(evaluate_model(baseline_logistic, X_test_comb,  y_test,  "baseline", "test"))


results_df = pd.DataFrame(rows)
results_df = results_df.set_index(["model", "split"])
display(results_df)

roc_auc  precision  recall      f1  accuracy
model    split                                              
baseline train   0.9419     0.8167  0.8324  0.8245    0.8694
         val     0.9007     0.7542  0.7662  0.7602    0.8216
         test    0.9061     0.7624  0.7638  0.7631    0.8238

## 5. Summary pivot

In [6]:
pivot = results_df.reset_index().pivot(index="split", columns="model")
# Reorder rows: train → val → test
pivot = pivot.loc[["train", "val", "test"]]
display(pivot)

,roc_auc,precision,recall,f1,accuracy
model,baseline,baseline,baseline,baseline,baseline
split,,,,,
train,0.9419,0.8167,0.8324,0.8245,0.8694
val,0.9007,0.7542,0.7662,0.7602,0.8216
test,0.9061,0.7624,0.7638,0.7631,0.8238


### 2b. Load SBERT classifier and pre-computed feature matrices

The SBERT features were encoded in `train_models.ipynb` and saved as `.npy` files.
Loading them here is instant (no encoder required) so this notebook stays fast.

In [7]:
# ADDED — load the SBERT Logistic Regression classifier from disk
sbert_logistic = load_object(os.path.join(MODELS_DIR, "sbert_logistic.pkl"))  # ADDED

# ADDED — load the pre-computed SBERT feature matrices (encoded in train_models.ipynb)
# These are plain numpy arrays: shape (n_samples, 768) = 2 * 384-dim MiniLM embeddings
X_train_sbert = np.load(os.path.join(MODELS_DIR, "sbert_X_train.npy"))  # ADDED
X_val_sbert   = np.load(os.path.join(MODELS_DIR, "sbert_X_val.npy"))    # ADDED
X_test_sbert  = np.load(os.path.join(MODELS_DIR, "sbert_X_test.npy"))   # ADDED

print("SBERT model and feature matrices loaded.")           # ADDED
print(f"  train shape: {X_train_sbert.shape}")            # ADDED
print(f"  val   shape: {X_val_sbert.shape}")              # ADDED
print(f"  test  shape: {X_test_sbert.shape}")             # ADDED

SBERT model and feature matrices loaded.
  train shape: (291897, 768)
  val   shape: (15363, 768)
  test  shape: (16172, 768)


### 4b. SBERT model evaluation

The SBERT model uses `paraphrase-MiniLM-L6-v2` semantic embeddings
(`|emb_q1 − emb_q2|` and `emb_q1 ⊙ emb_q2`) fed into a LogisticRegression.
Unlike the lexical models above, SBERT captures synonym and paraphrase
relationships invisible to BoW / TF-IDF.

In [8]:
# ADDED — evaluate the SBERT model on all three splits and append to results_df
sbert_rows = [  # ADDED
    evaluate_model(sbert_logistic, X_train_sbert, y_train, "sbert", "train"),  # ADDED
    evaluate_model(sbert_logistic, X_val_sbert,   y_val,   "sbert", "val"),    # ADDED
    evaluate_model(sbert_logistic, X_test_sbert,  y_test,  "sbert", "test"),   # ADDED
]  # ADDED

# Merge SBERT results into the main results table
sbert_df   = pd.DataFrame(sbert_rows).set_index(["model", "split"])  # ADDED
#results_df = pd.DataFrame(rows)
#results_df = pd.concat([results_df, sbert_df])                        # ADDED
display(sbert_df)                                                    # ADDED

roc_auc  precision  recall      f1  accuracy
model split                                              
sbert train   0.8988     0.7567  0.7430  0.7498    0.8172
      val     0.8939     0.7574  0.7277  0.7423    0.8136
      test    0.8983     0.7592  0.7353  0.7471    0.8151

### 5b. Summary pivot — all three models side by side

In [9]:
# ADDED — full three-model pivot: baseline | improved | sbert
full_pivot = sbert_df.reset_index().pivot(index="split", columns="model")  # ADDED
full_pivot = full_pivot.loc[["train", "val", "test"]]                         # ADDED
display(full_pivot)                                                            # ADDED

,roc_auc,precision,recall,f1,accuracy
model,sbert,sbert,sbert,sbert,sbert
split,,,,,
train,0.8988,0.7567,0.7430,0.7498,0.8172
val,0.8939,0.7574,0.7277,0.7423,0.8136
test,0.8983,0.7592,0.7353,0.7471,0.8151


### 2c. Load SBERT+Graph classifier and pre-computed feature matrices

The SBERT+Graph features (774 dims = SBERT 768 + graph 6) were computed in
`train_models.ipynb` and saved as `.npy` files.  The graph dictionaries
(`freq_dict`, `neighbor_dict`) are loaded from pickle — they are needed only
to verify shapes here; the heavy computation was done at training time.

In [10]:
# ADDED — load the SBERT+Graph LogisticRegression classifier
sbert_graph_logistic = load_object(os.path.join(MODELS_DIR, "sbert_graph_logistic.pkl"))  # ADDED

# ADDED — load the pre-computed SBERT+Graph feature matrices
# Shape: (n_samples, 774) = 768 SBERT dims + 6 graph dims
X_train_sg = np.load(os.path.join(MODELS_DIR, "sbert_graph_X_train.npy"))  # ADDED
X_val_sg   = np.load(os.path.join(MODELS_DIR, "sbert_graph_X_val.npy"))    # ADDED
X_test_sg  = np.load(os.path.join(MODELS_DIR, "sbert_graph_X_test.npy"))   # ADDED

print("SBERT+Graph model and feature matrices loaded.")     # ADDED
print(f"  train shape: {X_train_sg.shape}")               # ADDED
print(f"  val   shape: {X_val_sg.shape}")                 # ADDED
print(f"  test  shape: {X_test_sg.shape}")                # ADDED

SBERT+Graph model and feature matrices loaded.
  train shape: (291897, 774)
  val   shape: (15363, 774)
  test  shape: (16172, 774)


### 4c. SBERT+Graph model evaluation

This model adds **graph / magic features** on top of the SBERT embeddings:
question frequency (degree of the question node in the pair graph) and
neighbor intersection (number of questions both q1 and q2 are paired with).
These were the single biggest gain in virtually all top-10 Kaggle solutions.

In [11]:
# ADDED — evaluate the SBERT+Graph model on all three splits
sg_rows = [  # ADDED
    evaluate_model(sbert_graph_logistic, X_train_sg, y_train, "sbert_graph", "train"),  # ADDED
    evaluate_model(sbert_graph_logistic, X_val_sg,   y_val,   "sbert_graph", "val"),    # ADDED
    evaluate_model(sbert_graph_logistic, X_test_sg,  y_test,  "sbert_graph", "test"),   # ADDED
]  # ADDED

# Merge into the main results table
sg_df      = pd.DataFrame(sg_rows).set_index(["model", "split"])  # ADDED
display(sg_df)                                                 # ADDED

roc_auc  precision  recall      f1  accuracy
model       split                                              
sbert_graph train   0.9451     0.8457  0.7866  0.8151    0.8684
            val     0.9448     0.8522  0.7752  0.8119    0.8675
            test    0.9443     0.8499  0.7729  0.8096    0.8650

### 5c. Final summary pivot — all four models side by side

In [12]:
# ADDED — complete four-model pivot: baseline | improved | sbert | sbert_graph
final_pivot = sg_df.reset_index().pivot(index="split", columns="model")  # ADDED
final_pivot = final_pivot.loc[["train", "val", "test"]]                        # ADDED
display(final_pivot)                                                            # ADDED

,roc_auc,precision,recall,f1,accuracy
model,sbert_graph,sbert_graph,sbert_graph,sbert_graph,sbert_graph
split,,,,,
train,0.9451,0.8457,0.7866,0.8151,0.8684
val,0.9448,0.8522,0.7752,0.8119,0.8675
test,0.9443,0.8499,0.7729,0.8096,0.8650


## HYBRID MODEL

In [13]:
# #ADDING - Cell 1: Create the Hybrid Early Fusion Features
import scipy.sparse as sp

print("Concatenating Model A (handcrafted/combined) and Model B (SBERT) features...")

# #REPLACING - Cell 1: Memory-Efficient Early Fusion
import numpy as np

print("Creating a memory-efficient Hybrid model...")

# 1. Isolate just the handcrafted features (the last 9 columns of X_train_comb)
# Since X_train_comb was (BoW + handcrafted), we take the tail end.
num_handcrafted = 9 

# We convert them to dense because they are only 9 columns - very low memory impact
X_train_handcrafted = X_train_comb[:, -num_handcrafted:].toarray()
X_val_handcrafted   = X_val_comb[:, -num_handcrafted:].toarray()
X_test_handcrafted  = X_test_comb[:, -num_handcrafted:].toarray()

# 2. Concatenate with SBERT (Dense + Dense = Easy on Memory)
X_train_hybrid = np.hstack([X_train_handcrafted, X_train_sbert])
X_val_hybrid   = np.hstack([X_val_handcrafted, X_val_sbert])
X_test_hybrid  = np.hstack([X_test_handcrafted, X_test_sbert])

print(f"  New Hybrid train shape: {X_train_hybrid.shape}") 
# This should be (N_rows, 777) -- much smaller than 150,000 columns!

Concatenating Model A (handcrafted/combined) and Model B (SBERT) features...
Creating a memory-efficient Hybrid model...
  New Hybrid train shape: (291897, 777)


In [14]:
# #ADDING - Cell 2: Train the Hybrid Logistic Regression Model
from sklearn.linear_model import LogisticRegression

print("Training Early Fusion Logistic Regression...")

# We increase max_iter slightly to ensure the solver converges with the wider feature set
hybrid_logistic = LogisticRegression(max_iter=1000, random_state=123)
hybrid_logistic.fit(X_train_hybrid, y_train)

print("Training complete! Model is ready for interpretability analysis.")

Training Early Fusion Logistic Regression...
Training complete! Model is ready for interpretability analysis.


In [15]:
# #ADDING - Cell 3: Evaluate the Hybrid Model on all splits
hybrid_rows = [
    evaluate_model(hybrid_logistic, X_train_hybrid, y_train, "early_fusion", "train"),
    evaluate_model(hybrid_logistic, X_val_hybrid,   y_val,   "early_fusion", "val"),
    evaluate_model(hybrid_logistic, X_test_hybrid,  y_test,  "early_fusion", "test"),
]

# Display standalone hybrid results
hybrid_df = pd.DataFrame(hybrid_rows).set_index(["model", "split"])
display(hybrid_df)

roc_auc  precision  recall      f1  accuracy
model        split                                              
early_fusion train   0.9156     0.7745  0.7874  0.7809    0.8371
             val     0.9119     0.7741  0.7724  0.7733    0.8329
             test    0.9153     0.7793  0.7901  0.7847    0.8389

In [16]:
# #ADDING - Cell 4: Final summary pivot — all FIVE models side by side
# Safely combines all evaluation rows previously generated in the notebook
all_rows = rows + sbert_rows + sg_rows + hybrid_rows

ultimate_results_df = pd.DataFrame(all_rows).set_index(["model", "split"])

ultimate_pivot = ultimate_results_df.reset_index().pivot(index="split", columns="model")
# Reorder rows: train → val → test
ultimate_pivot = ultimate_pivot.loc[["train", "val", "test"]]
display(ultimate_pivot)

roc_auc                                  precision               \
model baseline early_fusion   sbert sbert_graph  baseline early_fusion   
split                                                                    
train   0.9419       0.9156  0.8988      0.9451    0.8167       0.7745   
val     0.9007       0.9119  0.8939      0.9448    0.7542       0.7741   
test    0.9061       0.9153  0.8983      0.9443    0.7624       0.7793   

                            recall                                        f1  \
model   sbert sbert_graph baseline early_fusion   sbert sbert_graph baseline   
split                                                                          
train  0.7567      0.8457   0.8324       0.7874  0.7430      0.7866   0.8245   
val    0.7574      0.8522   0.7662       0.7724  0.7277      0.7752   0.7602   
test   0.7592      0.8499   0.7638       0.7901  0.7353      0.7729   0.7631   

                                       accuracy                       \
model early_fusion   sbert sbert_graph baseline early_fusion   sbert   
split                                                                  
train       0.7809  0.7498      0.8151   0.8694       0.8371  0.8172   
val         0.7733  0.7423      0.8119   0.8216       0.8329  0.8136   
test        0.7847  0.7471      0.8096   0.8238       0.8389  0.8151   

                   
model sbert_graph  
split              
train      0.8684  
val        0.8675  
test       0.8650